In [1]:
import torch

# int8 conversion wraps, silently
print(torch.tensor([128.0]).to(torch.int8).item())      # -128

# but amax/127 can't get you there: it peaks just above 127, never near 127.5
w = torch.randn(100_000, 8)
q = w / (w.abs().amax(-1, keepdim=True) / 127)
print(q.abs().max().item())                              # 127.0000076


-128
127.00000762939453


In [2]:
import torch

def quantize(w, dim=-1):
    amax = w.abs().amax() if dim is None else w.abs().amax(dim=dim, keepdim=True)
    scale = (amax / 127).clamp(min=torch.finfo(w.dtype).tiny)
    return (w / scale).round().clamp(-127, 127).to(torch.int8), scale

a = torch.linspace(-0.1, 0.1, 9)
b = torch.linspace(-0.5, 0.5, 9)
w = torch.stack([a, b])

q, s = quantize(w)                       # per-channel
print("integers:\n", q)                  # the two rows are IDENTICAL
print("scales:", s.flatten())            # 7.87e-04 vs 3.94e-03
print("recovered:\n", (q * s))           # back to +-0.1 and +-0.5


integers:
 tensor([[-127,  -95,  -64,  -32,    0,   32,   64,   95,  127],
        [-127,  -95,  -64,  -32,    0,   32,   64,   95,  127]],
       dtype=torch.int8)
scales: tensor([0.0008, 0.0039])
recovered:
 tensor([[-0.1000, -0.0748, -0.0504, -0.0252,  0.0000,  0.0252,  0.0504,  0.0748,
          0.1000],
        [-0.5000, -0.3740, -0.2520, -0.1260,  0.0000,  0.1260,  0.2520,  0.3740,
          0.5000]])


In [3]:
qt, st = quantize(w, dim=None)           # per-tensor
print("integers:\n", qt)                 # row a is squashed to +-25
print("scale:", st)
print("recovered:\n", (qt * st))

integers:
 tensor([[ -25,  -19,  -13,   -6,    0,    6,   13,   19,   25],
        [-127,  -95,  -64,  -32,    0,   32,   64,   95,  127]],
       dtype=torch.int8)
scale: tensor(0.0039)
recovered:
 tensor([[-0.0984, -0.0748, -0.0512, -0.0236,  0.0000,  0.0236,  0.0512,  0.0748,
          0.0984],
        [-0.5000, -0.3740, -0.2520, -0.1260,  0.0000,  0.1260,  0.2520,  0.3740,
          0.5000]])


In [4]:
import torch
for dt in (torch.float32, torch.bfloat16, torch.float16):
    fi = torch.finfo(dt)
    print(f"{str(dt):<16} tiny={fi.tiny:<10.3e} eps={fi.eps:.3e}")


torch.float32    tiny=1.175e-38  eps=1.192e-07
torch.bfloat16   tiny=1.175e-38  eps=7.812e-03
torch.float16    tiny=6.104e-05  eps=9.766e-04


In [5]:
a = torch.linspace(-0.1, 0.1, 9)
b = torch.linspace(-0.0, 0.0, 9)
w = torch.stack([a, b])
q, s = quantize(w)     
print("integers:\n", q)                  # the two rows are IDENTICAL
print("scales:", s.flatten())            # 7.87e-04 vs 3.94e-03
print("recovered:\n", (q * s))          

integers:
 tensor([[-127,  -95,  -64,  -32,    0,   32,   64,   95,  127],
        [   0,    0,    0,    0,    0,    0,    0,    0,    0]],
       dtype=torch.int8)
scales: tensor([7.8740e-04, 1.1755e-38])
recovered:
 tensor([[-0.1000, -0.0748, -0.0504, -0.0252,  0.0000,  0.0252,  0.0504,  0.0748,
          0.1000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000]])


In [6]:
import torch

x     = torch.tensor([1000.0, 0.001])
x_hat = torch.tensor([1001.0, 0.002])

abs_err = (x_hat - x).abs()
rel_err = abs_err / x.abs()

print(abs_err)   # tensor([1.0000e+00, 1.0000e-03])
print(rel_err)
print(rel_err * 100)   # tensor([1.0000e-03, 1.0000e+00])


tensor([1.0000, 0.0010])
tensor([0.0010, 1.0000])
tensor([  0.1000, 100.0000])


In [7]:
def add_perc(x, p):
    return x + x * p/100
add_perc(1000, .1)

1001.0

In [8]:
add_perc(0.001, 100)

0.002

### 4 bit packing/unpacking

In [9]:
import torch

q = torch.tensor([-7, -1, 0, 5, 7], dtype=torch.int8)
for v in q.tolist():
    print(f"{v:>3} -> {v & 0xFF:08b}   top 4 bits are pure padding")


 -7 -> 11111001   top 4 bits are pure padding
 -1 -> 11111111   top 4 bits are pure padding
  0 -> 00000000   top 4 bits are pure padding
  5 -> 00000101   top 4 bits are pure padding
  7 -> 00000111   top 4 bits are pure padding


In [10]:
0xF

15

In [11]:
for v in range(-8, 8):
    nib = v & 0xF
    back = nib - 16 if nib >= 8 else nib          # sign-extend from 4 bits
    print(f"{v:>3}  {v & 0xFF:08b}  ->  {nib:04b} ({nib:>2})  ->  {back:>3}")


 -8  11111000  ->  1000 ( 8)  ->   -8
 -7  11111001  ->  1001 ( 9)  ->   -7
 -6  11111010  ->  1010 (10)  ->   -6
 -5  11111011  ->  1011 (11)  ->   -5
 -4  11111100  ->  1100 (12)  ->   -4
 -3  11111101  ->  1101 (13)  ->   -3
 -2  11111110  ->  1110 (14)  ->   -2
 -1  11111111  ->  1111 (15)  ->   -1
  0  00000000  ->  0000 ( 0)  ->    0
  1  00000001  ->  0001 ( 1)  ->    1
  2  00000010  ->  0010 ( 2)  ->    2
  3  00000011  ->  0011 ( 3)  ->    3
  4  00000100  ->  0100 ( 4)  ->    4
  5  00000101  ->  0101 ( 5)  ->    5
  6  00000110  ->  0110 ( 6)  ->    6
  7  00000111  ->  0111 ( 7)  ->    7


In [12]:
lo = torch.tensor([-7], dtype=torch.int8)
hi = torch.tensor([5], dtype=torch.int8)

l = (lo & 0xF).to(torch.uint8)
h = (hi & 0xF).to(torch.uint8)
byte = l | (h << 4)
print(f"lo {lo.item():>3} -> {l.item():04b}")
print(f"hi {hi.item():>3} -> {h.item():04b}")
print(f"byte       {byte.item():08b} = {byte.item()}")

# why the cast to uint8 comes BEFORE the shift:
print("9 << 4 in int8 :", (torch.tensor([9], dtype=torch.int8) << 4).item())
print("9 << 4 in uint8:", (torch.tensor([9], dtype=torch.uint8) << 4).item())


lo  -7 -> 1001
hi   5 -> 0101
byte       01011001 = 89
9 << 4 in int8 : -112
9 << 4 in uint8: 144


In [13]:
q = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7]], dtype=torch.int8)
print("even positions -> low nibbles:", q[..., 0::2].tolist())
print("odd  positions -> high nibbles:", q[..., 1::2].tolist())

lo, hi = q[..., 0::2], q[..., 1::2]
print("stack + flatten restores order:", torch.stack([lo, hi], dim=-1).flatten(-2).tolist())


even positions -> low nibbles: [[0, 2, 4, 6]]
odd  positions -> high nibbles: [[1, 3, 5, 7]]
stack + flatten restores order: [[0, 1, 2, 3, 4, 5, 6, 7]]


In [14]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src"))
sys.path.append(str(root / "src" / "video"))


In [15]:
from torch import Tensor
def pack_int4(q: Tensor) -> Tensor:
    """[..., n] int8 in [-7, 7] -> [..., n // 2] uint8, two values per byte.

    Masking to a nibble is what makes int4 actually save anything: without it a
    4-bit weight still occupies a whole int8 byte.
    """
    assert q.size(-1) % 2 == 0, "an odd row cannot be packed in pairs"
    lo = (q[..., 0::2] & 0xF).to(torch.uint8)
    hi = (q[..., 1::2] & 0xF).to(torch.uint8)
    return lo | (hi << 4)

def unpack_int4(p: Tensor) -> Tensor:
    """The inverse. The subtract is a 4-bit two's-complement sign extension."""
    lo, hi = (p & 0xF).to(torch.int8), (p >> 4).to(torch.int8)
    lo = torch.where(lo >= 8, lo - 16, lo)
    hi = torch.where(hi >= 8, hi - 16, hi)
    return torch.stack([lo, hi], dim=-1).flatten(-2)
q = torch.arange(-7, 8, dtype=torch.int8).repeat(2)[:16].reshape(2, 8)
p = pack_int4(q)
print(f"{q.numel()} int8 bytes -> {p.numel()} uint8 bytes")
print("exact round trip:", torch.equal(unpack_int4(p), q))


16 int8 bytes -> 8 uint8 bytes
exact round trip: True


In [16]:
import torch
from quantize import block, acc_dtype # type: ignore

for bits in (4, 5, 6, 8):
    k, nb = block(bits)
    dt = acc_dtype(bits)
    per_elem = torch.tensor([], dtype=dt).element_size()
    print(f"int{bits}: {k} values -> {k*bits:>2} bits -> {str(dt):<12} "
          f"{per_elem}B/elem vs 8B for int64")


int4: 2 values ->  8 bits -> torch.int16  2B/elem vs 8B for int64
int5: 8 values -> 40 bits -> torch.int64  8B/elem vs 8B for int64
int6: 4 values -> 24 bits -> torch.int32  4B/elem vs 8B for int64
int8: 1 values ->  8 bits -> torch.int16  2B/elem vs 8B for int64
